In [ ]:
# @title Earnings Scanner (Silent Mode + Clean Copy List)
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta, date
import logging

# --- 1. SILENCE YFINANCE LOGGERS ---
# We set the yfinance logger to only show "CRITICAL" issues, suppressing Errors and Warnings
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)
yf_logger.disabled = True

# --- 2. DATE SETUP ---
today = datetime.now().date()

# Find next Monday
days_ahead = 0 - today.weekday()
if days_ahead <= 0: 
    days_ahead += 7
next_monday = today + timedelta(days=days_ahead)

# Target Window: 7 to 12 days after next Monday
target_start_date = next_monday + timedelta(days=7)
target_end_date = next_monday + timedelta(days=12)

print(f"--- SCAN CONFIGURATION ---")
print(f"Current System Date: {today}")
print(f"Entry Date (Mon):    {next_monday}")
print(f"Earnings Window:     {target_start_date} to {target_end_date}")
print(f"------------------------\n")

# --- 3. WATCHLIST ---
tickers_string = "A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EXC EXPE F FANG FAST FCX FDX FE FITB FOXA FSLR FTI FTV GD GE GILD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HCA HD HIG HLT HOG HOLX HON HPE HPQ IBM ICE INTC IP IRM IVZ JCI JD JNJ JPM K KHC KMI KO KR KSS LEN LI LKQ LLY LNC LOW LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OKE OMC ON ORCL OXY PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHW SEDG SHOP SIRI SLB SMCI SNAP SNOW SO SOFI SPG STX SWKS SYF SYY T TAP TCOM TDOC TFC TGT TJX TMO TMUS TPR TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNH UNP UPS UPST URBN USB V VALE VFC VTR VZ WAB WBD WDC WFC WM WMB WMT WU WY WYNN XEL YELP ZION ZM"
ticker_list = list(set(tickers_string.split())) 
results = []

print(f"Scanning {len(ticker_list)} stocks... (Silent Mode)")

# --- 4. SCANNING LOOP ---
for symbol in ticker_list:
    try:
        stock = yf.Ticker(symbol)
        
        # Robust Date Extraction
        cal = stock.calendar
        dates_to_check = []
        
        if cal is not None:
            if isinstance(cal, dict) and 'Earnings Date' in cal:
                dates_to_check = list(cal['Earnings Date'])
            elif isinstance(cal, pd.DataFrame):
                dates_to_check = [v for v in cal.values.flatten() if isinstance(v, (date, datetime, pd.Timestamp))]
            elif isinstance(cal, list):
                dates_to_check = cal

        # Check Logic
        match_found = False
        confirmed_date = None
        
        for d in dates_to_check:
            if hasattr(d, 'date'): check_date = d.date()
            else: check_date = d
            
            if target_start_date <= check_date <= target_end_date:
                match_found = True
                confirmed_date = check_date
                break
        
        if match_found:
            info = stock.info
            beta = info.get('beta', 0)
            vol = info.get('averageVolume', 0)
            
            if vol is not None and vol > 300000:
                results.append({
                    'Ticker': symbol,
                    'Earnings Date': confirmed_date,
                    'Beta': beta,
                    'Avg Volume': vol
                })
    except Exception:
        # Since logger is disabled, we just pass silently
        pass

# --- 5. RESULTS & COPY PASTE LIST ---
if len(results) > 0:
    df = pd.DataFrame(results)
    # Sort by Beta (Highest first)
    df_sorted = df.sort_values(by='Beta', ascending=False).head(20)
    
    # Clean Format
    df_display = df_sorted.copy()
    df_display['Avg Volume'] = df_display['Avg Volume'].apply(lambda x: "{:,.0f}".format(x) if x else "N/A")
    
    print(f"\nSUCCESS: Found {len(df)} stocks. Top 20 Candidates (High Volatility First):")
    display(df_display)
    
    # --- COPY PASTE SECTION ---
    print("\n--- COPY FOR ANALYSIS ---")
    tickers = df_sorted['Ticker'].tolist()
    
    # Batch 1 (First 10)
    batch1 = tickers[:10]
    print(", ".join(batch1))
    
    # Batch 2 (Next 10)
    if len(tickers) > 10:
        batch2 = tickers[10:20]
        print(", ".join(batch2))
        
else:
    print(f"\nNo stocks found for earnings between {target_start_date} and {target_end_date}.")